# 02 — Retrieval Eval: Embedding-only vs Hybrid (BM25 + Embeddings)

Compares plain embedding similarity search against the hybrid retrieval
path (`retrieval.py`'s BM25 + Chroma merged via Reciprocal Rank Fusion) on
a small set of hand-written queries with a known expected `form_id` (see
APPROACH.md §3.6, §6 step 6).

Metrics:
- **Hit@k** — did the expected form_id appear anywhere in the top-k chunks?
- **MRR** (Mean Reciprocal Rank) — how high did it rank when it did hit?
  1/rank per query, averaged; rewards ranking the right form_id *first*,
  not just somewhere in the top-k.

Requires Ollama running locally (embeddings need `nomic-embed-text` pulled).

In [ ]:
import sys, os, shutil, tempfile
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
import ingestion
import extraction
import structured_store
import vectorstore
import router
import retrieval
import llm_client
from chunking import build_chunks

SAMPLE_DIR = os.path.abspath("../data/sample_forms")
TOP_K = 3

if not llm_client.is_available(llm_client.DEFAULT_EMBED_MODEL):
    print(f"⚠️  Ollama / {llm_client.DEFAULT_EMBED_MODEL} not reachable.")
    print("   Run `ollama serve` and `ollama pull nomic-embed-text` first.")
else:
    print("✅ Ollama reachable, embedding model available.")

## Build a fresh in-memory/temp-dir store from the sample forms

In [ ]:
conn = structured_store.connect(":memory:")
structured_store.init_db(conn)

tmp_chroma_dir = tempfile.mkdtemp(prefix="chroma_eval_")
client = vectorstore.get_client(persist_dir=tmp_chroma_dir)
collection = vectorstore.get_collection(client)

loaded = []
for raw in ingestion.ingest_directory(SAMPLE_DIR):
    try:
        extracted = extraction.extract_form(raw)
        structured_store.insert_form(conn, extracted)
        vectorstore.delete_form(collection, extracted.form_id)
        vectorstore.add_chunks(collection, build_chunks(extracted))
        loaded.append(extracted.form_id)
    except (extraction.ExtractionError, vectorstore.VectorStoreError) as e:
        print(f"Failed to load {raw.form_id}: {e}")

print(f"Loaded {len(loaded)} form(s) into structured + vector stores: {loaded}")

## Test queries with known expected `form_id`

Each query targets language that's distinctive to one form's free-text
field (`remarks` / `doctors_notes`), so there's a single unambiguous
correct answer to check retrieval against.

In [ ]:
TEST_QUERIES = [
    ("Which applicant had prior credit defaults?", "membership_002"),
    ("Who was approved for the premium membership tier?", "membership_001"),
    ("Which patient had an angioplasty procedure?", "hospital_001"),
    ("Which patient needs a follow-up X-ray in a few weeks?", "hospital_002_scanned"),
    ("Who was advised to follow a low-sodium diet?", "hospital_001"),
    ("Which applicant can reapply after 12 months?", "membership_002"),
]
pd.DataFrame(TEST_QUERIES, columns=["query", "expected_form_id"])

## Run both retrieval paths per query

In [ ]:
def rank_of(expected_form_id, chunks):
    """1-indexed rank of the first chunk matching expected_form_id, or None if absent."""
    for i, c in enumerate(chunks, start=1):
        if c.form_id == expected_form_id:
            return i
    return None


rows = []
for query, expected_form_id in TEST_QUERIES:
    # --- Embedding-only ---
    embed_hits = vectorstore.similarity_search(collection, query, top_k=TOP_K, where=None)
    embed_rank = rank_of(expected_form_id, embed_hits)

    # --- Hybrid (BM25 + embeddings via RRF), forced multi_form_semantic route ---
    decision = router.RouteDecision(route=router.RouteType.multi_form_semantic, detection_method="forced")
    hybrid_result = retrieval.retrieve(decision, query, conn, collection, top_k=TOP_K)
    hybrid_rank = rank_of(expected_form_id, hybrid_result.chunks)

    rows.append({
        "query": query,
        "expected_form_id": expected_form_id,
        "embed_top_k_form_ids": [c.form_id for c in embed_hits],
        "embed_rank": embed_rank,
        "hybrid_top_k_form_ids": [c.form_id for c in hybrid_result.chunks],
        "hybrid_rank": hybrid_rank,
    })

results_df = pd.DataFrame(rows)
results_df

## Score: Hit@k and MRR for each method

In [ ]:
def hit_at_k(ranks):
    return sum(r is not None for r in ranks) / len(ranks)

def mrr(ranks):
    return sum((1.0 / r) if r else 0.0 for r in ranks) / len(ranks)

embed_ranks = results_df["embed_rank"].tolist()
hybrid_ranks = results_df["hybrid_rank"].tolist()

summary = pd.DataFrame({
    "method": ["embedding-only", "hybrid (BM25 + embeddings, RRF)"],
    f"hit@{TOP_K}": [hit_at_k(embed_ranks), hit_at_k(hybrid_ranks)],
    "MRR": [mrr(embed_ranks), mrr(hybrid_ranks)],
})
summary

In [ ]:
shutil.rmtree(tmp_chroma_dir, ignore_errors=True)
conn.close()
print("Cleaned up temp Chroma dir and closed connection.")

## Notes

- At this corpus size (4 forms, ~1 chunk each), both methods will likely
  score similarly or even identically — hybrid retrieval's advantage shows
  up more clearly as the corpus grows and pure-embedding search starts
  surfacing semantically-similar-but-wrong forms. Re-run this notebook
  after adding more sample forms via `_generate_samples.py`-style
  generation to see the gap widen.
- If a query consistently ranks wrong in **both** methods, that's a
  chunking or query-phrasing problem, not a retrieval-algorithm problem —
  check `chunking.build_chunks()`'s output for that form first.
- The optional cross-encoder reranker (APPROACH.md §3.6 step 3, not yet
  wired into `retrieval.py`) would slot in right after the RRF merge in
  `_multi_form_semantic` — a good next experiment once this baseline
  comparison is captured, and worth comparing against here before deciding
  if it's worth the extra latency for your corpus size.